In [1]:
import pandas as pd
import numpy as np

In [2]:
movies = pd.read_csv('movie.csv')
ratings = pd.read_csv('rating.csv')

In [3]:
movie_data = pd.merge(movies , ratings , on='movieId')

In [4]:
movie_data.head()

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3,4.0,1999-12-11 13:36:47
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,6,5.0,1997-03-13 17:50:52
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,8,4.0,1996-06-05 13:37:51
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,10,4.0,1999-11-25 02:44:47
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,11,4.5,2009-01-02 01:13:41


In [5]:
rating_mean = pd.DataFrame(movie_data.groupby('title')['rating'].mean())

In [6]:
rating_mean.head()

,rating
title,
#chicagoGirl: The Social Network Takes on a Dictator (2013),3.666667
$ (Dollars) (1971),2.833333
$5 a Day (2008),2.871795
$9.99 (2008),3.009091
$ellebrity (Sellebrity) (2012),2.000000


In [7]:
rating_mean['rating_mean_count'] = pd.DataFrame(movie_data.groupby('title')['rating'].count())

In [8]:
rating_mean.head()

,rating,rating_mean_count
title,,
#chicagoGirl: The Social Network Takes on a Dictator (2013),3.666667,3
$ (Dollars) (1971),2.833333,24
$5 a Day (2008),2.871795,39
$9.99 (2008),3.009091,55
$ellebrity (Sellebrity) (2012),2.000000,2


In [9]:
rating_mean['rating'] = round(rating_mean['rating'] , 1)

In [10]:
rating_mean.head()

,rating,rating_mean_count
title,,
#chicagoGirl: The Social Network Takes on a Dictator (2013),3.7,3
$ (Dollars) (1971),2.8,24
$5 a Day (2008),2.9,39
$9.99 (2008),3.0,55
$ellebrity (Sellebrity) (2012),2.0,2


In [11]:
stats = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

In [12]:
stats

,movieId,avg_rating,rating_count
0,1,3.921240,49695
1,2,3.211977,22243
2,3,3.151040,12735
3,4,2.861393,2756
4,5,3.064592,12161
...,...,...,...
26739,131254,4.000000,1
26740,131256,4.000000,1
26741,131258,2.500000,1
26742,131260,3.000000,1


In [13]:
stats['avg_rating'] = round(stats['avg_rating'] , 1)

In [14]:
stats.head()

,movieId,avg_rating,rating_count
0,1,3.9,49695
1,2,3.2,22243
2,3,3.2,12735
3,4,2.9,2756
4,5,3.1,12161


In [15]:
movies = movies.merge(stats , on='movieId' , how='left')

In [16]:
movies['avg_rating'] = movies['avg_rating'].fillna(0)
movies['rating_count'] = movies['rating_count'].fillna(0)

In [17]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movieId       27278 non-null  int64  
 1   title         27278 non-null  object 
 2   genres        27278 non-null  object 
 3   avg_rating    27278 non-null  float64
 4   rating_count  27278 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.0+ MB


In [18]:
movies.head()

,movieId,title,genres,avg_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,49695.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.2,22243.0
2,3,Grumpier Old Men (1995),Comedy|Romance,3.2,12735.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.9,2756.0
4,5,Father of the Bride Part II (1995),Comedy,3.1,12161.0


In [19]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

In [20]:
movies.head()

,movieId,title,genres,avg_rating,rating_count,year,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,49695.0,1995,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.2,22243.0,1995,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,3.2,12735.0,1995,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.9,2756.0,1995,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,3.1,12161.0,1995,Father of the Bride Part II


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
movies['search_text'] = movies['clean_title'] + ' ' + movies['genres'].str.replace('|', ' ')

In [23]:
movies.head()

,movieId,title,genres,avg_rating,rating_count,year,clean_title,search_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,49695.0,1995,Toy Story,Toy Story Adventure Animation Children Comedy ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.2,22243.0,1995,Jumanji,Jumanji Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,3.2,12735.0,1995,Grumpier Old Men,Grumpier Old Men Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.9,2756.0,1995,Waiting to Exhale,Waiting to Exhale Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,3.1,12161.0,1995,Father of the Bride Part II,Father of the Bride Part II Comedy


In [24]:
movies['search_text'][0]

'Toy Story Adventure Animation Children Comedy Fantasy'

In [25]:
vectorizer = TfidfVectorizer(ngram_range=(1 , 2) , analyzer='word')
tfidf_matrix = vectorizer.fit_transform(movies['search_text'])

In [90]:
def search(query , top_n = 10  , genre_filter=None):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec , tfidf_matrix).flatten()

    movies['score'] = scores

    results = movies[movies['score'] > 0.3].copy()

    if genre_filter:
        results = results[results['genres'].str.contains(genre_filter, case=False)]

    results['final_score'] = (
        results['score'] * 0.6 + (results['avg_rating'] / 5) * 0.25 + (np.log1p(results['rating_count']) / 10) * 0.15
    ) 
    
    return results.sort_values('final_score' , ascending=False).head(top_n)

In [91]:
result = search("north" , genre_filter='comedy')

In [92]:
result

,movieId,title,genres,avg_rating,rating_count,year,clean_title,search_text,score,final_score
501,505,North (1994),Comedy,2.5,1293.0,1994,North,North Comedy,0.602689,0.594096
6106,6205,Mr. North (1988),Comedy|Drama,2.8,43.0,1988,Mr. North,Mr. North Comedy Drama,0.420100,0.448823
3416,3506,North Dallas Forty (1979),Comedy|Drama,3.2,708.0,1979,North Dallas Forty,North Dallas Forty Comedy Drama,0.314555,0.447191
7494,7816,North Beach (2000),Comedy|Drama,3.5,3.0,2000,North Beach,North Beach Comedy Drama,0.416861,0.445911
15613,79505,North (Nord) (2009),Comedy|Drama,3.3,18.0,2009,North (Nord),North (Nord) Comedy Drama,0.382335,0.438567
16592,83835,"Frozen North, The (1922)",Comedy,3.3,9.0,1922,"Frozen North, The","Frozen North, The Comedy",0.390937,0.434101
6345,6455,North to Alaska (1960),Comedy|Western,3.3,130.0,1960,North to Alaska,North to Alaska Comedy Western,0.318598,0.429287
2593,2679,Finding North (1998),Comedy|Drama|Romance,2.7,52.0,1998,Finding North,Finding North Comedy Drama Romance,0.389203,0.428076


In [40]:
r = result['score']*0.6

In [41]:
r

0        0.241641
3027     0.241641
4833     0.362051
5744     0.262487
15401    0.219159
15773    0.236332
16216    0.208872
19186    0.246282
21981    0.213743
24458    0.158496
24460    0.157837
25388    0.148717
25461    0.161901
25463    0.177680
Name: score, dtype: float64

In [39]:
result

,movieId,title,genres,avg_rating,rating_count,year,clean_title,search_text,score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.9,49695.0,1995,Toy Story,Toy Story Adventure Animation Children Comedy ...,0.402734
3027,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.8,22770.0,1999,Toy Story 2,Toy Story 2 Adventure Animation Children Comed...,0.402734
4833,4929,"Toy, The (1982)",Comedy,2.7,706.0,1982,"Toy, The","Toy, The Comedy",0.603418
5744,5843,Toy Soldiers (1991),Action|Drama,3.0,709.0,1991,Toy Soldiers,Toy Soldiers Action Drama,0.437479
15401,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,4.0,5781.0,2010,Toy Story 3,Toy Story 3 Adventure Animation Children Comed...,0.365264
15773,80141,"Christmas Toy, The (1986)",Children|Musical,3.2,12.0,1986,"Christmas Toy, The","Christmas Toy, The Children Musical",0.393887
16216,81981,"Toy, The (Le jouet) (1976)",Comedy,3.5,16.0,1976,"Toy, The (Le jouet)","Toy, The (Le jouet) Comedy",0.348120
19186,95446,Tin Toy (1988),Animation|Children,3.4,37.0,1988,Tin Toy,Tin Toy Animation Children,0.410470
21981,106022,Toy Story of Terror (2013),Animation|Children|Comedy,3.6,60.0,2013,Toy Story of Terror,Toy Story of Terror Animation Children Comedy,0.356238
24458,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,3.3,12.0,2011,Toy Story Toons: Hawaiian Vacation,Toy Story Toons: Hawaiian Vacation Adventure A...,0.264161


In [42]:
a = (result['avg_rating']/5) * 0.25

In [43]:
a

0        0.195
3027     0.190
4833     0.135
5744     0.150
15401    0.200
15773    0.160
16216    0.175
19186    0.170
21981    0.180
24458    0.165
24460    0.150
25388    0.110
25461    0.130
25463    0.155
Name: avg_rating, dtype: float64

In [45]:
l = (np.log1p(result['rating_count']) / 10) * 0.15

In [46]:
l

0        0.162205
3027     0.150499
4833     0.098415
5744     0.098479
15401    0.129938
15773    0.038474
16216    0.042498
19186    0.054564
21981    0.061663
24458    0.038474
24460    0.035968
25388    0.016479
25461    0.024142
25463    0.044167
Name: rating_count, dtype: float64